#GenerateExperiment.ipynb

#DESCRIPTION

## This notebook the Experiment Setup entry point

---

This notebook is the entry point for initializing a modular experiment pipeline. It emphasizes a clean, reproducible setup: minimal manual edits, clear separation of configuration from logic, and version-pinned file downloads. The goal is to streamline the workflow—select modules, configure paths, and prepare the environment with minimal friction.

All implementation details are encapsulated in external modules. This notebook guides you through a standard sequence: configure once, select components, inject paths, and stage dependencies. Edit only the cells Experiment Variables and SELECT FILES TO USE; everything else should run as-is.


####Run Order

1.   Set up experiment environment based on the selected root path, chosen modules, and define commit hash.

2.   Stage files and folders, creates only the required folder structure for the modules chosen.

3.   Downloads only necessary files from GitHub based on selected modules and injects paths into configuration files.

4.   Finalize setup, it validates file existence, resolve dependencies, and prepare for pipeline execution.





#Experiment Variables

Define the variable for configuration you want to be used in the experience.




In [ ]:
## Experiment variables

# Enable pose-derived metrics processing
POSE_SCORING = False  # include pose‑derived metrics from SLEAP

# Stimulus alignment configuration
ALIGNMENT_COL         = "VisualStim"  # column holding stimulus pulses (0→1)
STIMULUS_NUMBER       = 20            # expected onsets per run
STIMULUS_DURATION_SEC = 0.5           # stimulus length (sec)
EXPECTED_STIMULUS     = STIMULUS_NUMBER + 3  # extra events (e.g. lights‑off)


# Timing & arena dimensions
FRAME_RATE      = 60   # frames per second
ARENA_WIDTH_MM  = 30   # arena width (millimetres)
ARENA_HEIGHT_MM = 30   # arena height (millimetres)

# Experimental periods durations (sec)
EXPERIMENTAL_PERIODS = {
    "Baseline":    {"duration_sec": 300},
    "Stimulation": {"duration_sec": 300},
    "Recovery":    {"duration_sec": 300},
}

# Filename and grouping metadata
FILENAME_STRUCTURE = [  # order of fields in scored filenames
    "Experimenter", "Genotype", "Protocol", "Sex", "Age",
    "Setup", "Camera", "Date", "FlyID", "Extension",
]

# SELECT THE RUN FILES YOU WANT TO USE


In [ ]:
## IF true the script will download the files and its dependencies

BEHAVIOR_SCORING_RUN = True
CREATE_DATAFRAMES_RUN = False

##Plotter_Ethograms = True (needs to be discussed)

#CHOOSE EXPERIMENT FOLDER


##Script

In [ ]:
from google.colab import drive
from IPython.display import display, clear_output
from ipyfilechooser import FileChooser
import ipywidgets as widgets
import os

# Mount Drive (optional if not mounted)
try:
    drive.mount('/content/drive')
except Exception:
    pass

chooser = FileChooser('/content/drive/MyDrive/Matheus_e_Rodrigo/')
chooser.title = 'Select the experiment folder'
chooser.show_only_dirs = True
##display(chooser)

continue_button = widgets.Button(description="Continue")
output = widgets.Output()


# Global variable to save folder path
selected_experiment_folder = None

def on_continue_clicked(b):
    global selected_experiment_folder
    with output:
        clear_output()
        if not chooser.selected or not os.path.isdir(chooser.selected):
            print("⚠️ Please select a valid experiment folder before continuing.")
            selected_experiment_folder = None
        else:
            selected_experiment_folder = chooser.selected
            print(f"✅ Folder selected: {selected_experiment_folder}")



continue_button.on_click(on_continue_clicked)
##display(continue_button, output)


Mounted at /content/drive


##Display

In [ ]:
display(chooser)
display(continue_button, output)

FileChooser(path='/content/drive/MyDrive/Matheus_e_Rodrigo', filename='', title='Select the experiment folder'…

Button(description='Continue', style=ButtonStyle())

Output()

# SCRIPT (DO NOT CHANGE)

In [ ]:
import os, json, io, nbformat, shutil, subprocess, regex, datetime

# =========================================================
# DEPENDENCIES configuration dictionary
# This maps each "run type" (e.g., CreateDataFrames or BehaviorScoring)
# to:
#   - a flag (True/False) that indicates if this run should be executed
#   - a dictionary of which files to copy from the cloned repository
#     into the experiment folder for that run
# =========================================================

DEPENDENCIES = {
    'CREATE_DATAFRAMES_RUN': {
        'flag': CREATE_DATAFRAMES_RUN,
        'files_to_copy': {
            'CreateDataFrames': ['CreateDataFramesFunctions.py', 'CreateDataFramesMain.py', '__init__.py'],
            'Config': [
                'ExperimentConfig.py',
                'CreateDataFramesConfig.py',
                'PathConfig.py',
                'BehaviorScoringColabConfig.py',
                '__init__.py',
            ],
        },
    },
    'BEHAVIOR_SCORING_RUN': {
        'flag': BEHAVIOR_SCORING_RUN,
        'files_to_copy': {
            'BehaviorScoring': ['BehaviorScoringFunctions.py', 'BehaviorScoringMain.py', '__init__.py'],
            'Config': [
                'ExperimentConfig.py',
                'BehaviorScoringConfig.py',
                'PathConfig.py',
                'BehaviorScoringColabConfig.py',
                '__init__.py',
            ],
        },
    },
}

# =========================================================
# Repository URL and temporary clone path
# =========================================================

GIT_REPO_URL = 'https://github.com/rodrigprogram9/experiencias.git' ## IN THE FINAL VERSION IT WILL HAVE THE LAB GIT HUB
TMP_CLONE_PATH = '/tmp/experiencias_repo' ## CAN BE MAINTAINED; WAS JUST NAME BASED ON MY GIT REPOSITORY


# =========================================================
# FUNCTION: clone_repo_once
# Clones the repository fresh every time, removing old copies,
# and logs commit information (date/time, hash, file list) to
# "Protocols/CommitHash.txt" inside the experiment folder.
# =========================================================

def clone_repo_once(experimental_folder):
    # Remove any previous clone in /tmp to ensure fresh copy
    if os.path.exists(TMP_CLONE_PATH):
        print(f"🧹 Removing existing repo at {TMP_CLONE_PATH} ...")
        shutil.rmtree(TMP_CLONE_PATH)

    # Clone the repository
    print(f"Cloning repository from {GIT_REPO_URL} into {TMP_CLONE_PATH} ...")
    subprocess.run(['git', 'clone', GIT_REPO_URL, TMP_CLONE_PATH], check=True)
    print("✅ Repository cloning finished.")


    # Retrieve current commit hash
    commit_hash = subprocess.run(
        ['git', '-C', TMP_CLONE_PATH, 'rev-parse', 'HEAD'],
        stdout=subprocess.PIPE, text=True, check=True
    ).stdout.strip()

    # Get a list of tracked files in the repository
    file_list = subprocess.run(
        ['git', '-C', TMP_CLONE_PATH, 'ls-files'],
        stdout=subprocess.PIPE, text=True, check=True
    ).stdout.strip().split("\n")

    # Format file list
    file_list_str = " | ".join(file_list)

    # Get current datetime
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Ensure the Protocols folder exists and defines the Path of the log file
    log_dir = os.path.join(experimental_folder, "Protocols")
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, "CommitHash.txt")

    # Append commit info + GitHub download instructions
    with open(log_file, "a", encoding="utf-8") as f:
        file_list_str = "\n".join(file_list)
        output_str = (f"Access date: {now}\n"
              f"Commit Hash: {commit_hash}\n"
              f"List of downloaded files:\n"
              f"{file_list_str}\n")

        f.write(output_str)
        f.write(
            "\n--- How to download this exact version from GitHub ---\n"
            f"1. Open this URL in your browser:\n"
            f"   {GIT_REPO_URL.replace('.git', '')}/tree/{commit_hash}\n"
            "2. Click the green 'Code' button.\n"
            "3. Select 'Download ZIP'.\n"
            "4. Unzip the downloaded file to access the files for this commit.\n"
            "-------------------------------------------------------\n\n"
        )

    # Display information in the console
    print(f"📌 Commit hash: {commit_hash}")
    print(f"💾 Saved clone info to {log_file}")



# =========================================================
# FUNCTION: clone_behavior_scoring_notebook
# Copies "BehaviorScoringRun.ipynb" from the cloned repo
# to Protocols/Codes if it doesn't already exist.
# =========================================================

def clone_behavior_scoring_notebook(experimental_folder):
    src_nb = os.path.join(TMP_CLONE_PATH, 'BehaviorScoringRun.ipynb')
    dest_nb = os.path.join(experimental_folder, 'Protocols', 'Codes', 'BehaviorScoringRun.ipynb')

    # Skip if already exists
    if os.path.exists(dest_nb):
        print(f"✔️ BehaviorScoringRun.ipynb already exists in {dest_nb}, skipping copy.")
        return
    # Copy if source exists
    if os.path.exists(src_nb):
        os.makedirs(os.path.dirname(dest_nb), exist_ok=True)
        shutil.copy2(src_nb, dest_nb)
        print(f"Copied BehaviorScoringRun.ipynb to {dest_nb}")
    else:
        print(f"⚠️ BehaviorScoringRun.ipynb not found in cloned repo, cannot copy.")



# =========================================================
# FUNCTION: create_experiment_structure
# Creates the base experiment directory structure,
# plus additional folders for active runs.
# =========================================================

def create_experiment_structure(base_path):
    # Always-created folders, creates the folder structure
    always_folders = [
        os.path.join(base_path, "Protocols", "Codes", "Config"),
        os.path.join(base_path, "RawData"),
        os.path.join(base_path, "PostProcessing", "Arenas"),
        os.path.join(base_path, "PostProcessing", "CropVideo"),
        os.path.join(base_path, "PostProcessing", "Tracked"),
        os.path.join(base_path, "PostProcessing", "Pose"),
        os.path.join(base_path, "PostProcessing", "Scored"),
        os.path.join(base_path, "PostProcessing", "ScoredPose"),
        os.path.join(base_path, "PostProcessing", "ScoredError"),
        os.path.join(base_path, "Analysis", "DataFrames"),
        os.path.join(base_path, "Analysis", "ZoomDataFrames"),
        os.path.join(base_path, "Analysis", "Plots"),
    ]

    for folder in always_folders:
        os.makedirs(folder, exist_ok=True)

    # Conditional: Create BehaviorScoring folder
    if DEPENDENCIES.get('BEHAVIOR_SCORING_RUN', {}).get('flag', False):
        os.makedirs(os.path.join(base_path, "Protocols", "Codes", "BehaviorScoring"), exist_ok=True)
        print("✅ Created BehaviorScoring folder and Config subfolder")

    # Conditional: Create CreateDataFrames folder
    if DEPENDENCIES.get('CREATE_DATAFRAMES_RUN', {}).get('flag', False):
        os.makedirs(os.path.join(base_path, "Protocols", "Codes", "CreateDataFrames"), exist_ok=True)
        print("✅ Created CreateDataFrames folder and Config subfolder")

    print("✅ Experiment folder structure created")


# =========================================================
# FUNCTION: copy_files_for_run
# Copies files listed in DEPENDENCIES for the active run.
# Skips if file already exists.
# =========================================================

def copy_files_for_run(experimental_folder, run_info):
    base_codes_path = os.path.join(experimental_folder, 'Protocols', 'Codes')

    for subfolder, file_list in run_info['files_to_copy'].items():
        if not file_list:
            continue

        # Determine destination folder path based on run and subfolder
        if subfolder == 'Config':
            dest_folder = os.path.join(base_codes_path, "Config")
        else:
          if run_info['flag'] == DEPENDENCIES['BEHAVIOR_SCORING_RUN']['flag']:
            dest_folder = os.path.join(base_codes_path, "BehaviorScoring")
          elif run_info['flag'] == DEPENDENCIES['CREATE_DATAFRAMES_RUN']['flag']:
            dest_folder = os.path.join(base_codes_path, "CreateDataFrames")
          else:
            dest_folder = base_codes_path

        os.makedirs(dest_folder, exist_ok=True)

        for filename in file_list:
            src_path = os.path.join(TMP_CLONE_PATH, filename)
            dest_path = os.path.join(dest_folder, filename)

            # Skip if file already exists
            if os.path.exists(dest_path):
                print(f"✔️ {filename} already exists in {dest_folder}, skipping copy.")
                continue

            # Copy if found in cloned repo
            if os.path.exists(src_path):
                shutil.copy2(src_path, dest_path)
                print(f"Copied {filename} to {dest_folder}")
            else:
                print(f"⚠️ Warning: {filename} not found in cloned repo.")


# =========================================================
# FUNCTION: inject_exp_root_in_pathconfig
# Replaces placeholder "_PLACEHOLDER_EXPERIMENT_ROOT_"
# inside PathConfig.py with the actual experiment folder path.
# =========================================================

def inject_exp_root_in_pathconfig(path_config_path, experimental_folder):
    if not os.path.exists(path_config_path):
        print(f"⚠️ PathConfig.py not found at {path_config_path}, skipping injection.")
        return

    with open(path_config_path, "r") as f:
        content = f.read()

    new_content = content.replace("_PLACEHOLDER_EXPERIMENT_ROOT_", experimental_folder.replace("\\", "/"))

    with open(path_config_path, "w") as f:
        f.write(new_content)

    print(f"✅ Injected experiment root path into PathConfig.py")



# =========================================================
# FUNCTION: inject_code_from_ipynb
# Reads this notebook (.ipynb), extracts code from a
# specific cell, experiment variables cell, and injects it into a Python file after a tag (Experiment Config).
#Parameters: (found on the on_continue_clicked function, because the py_path depdends on the selected_folder)
        #ipynb_path (str): Path to the .ipynb notebook.
        #py_path (str): Path to the target .py file.
        #cell_index (int): Index of the code cell to extract (0-based).
        #insert_after_tag (str): Tag string to insert after in the .py file.
# =========================================================

def inject_code_from_ipynb(ipynb_path, py_path, cell_index, insert_after_tag):
    if not os.path.exists(ipynb_path):
        print(f"⚠️ Notebook file {ipynb_path} not found.")
        return

    # Load the notebook
    with open(ipynb_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    # Get the list of cells
    cells = notebook.get("cells", [])
    if cell_index >= len(cells) or cells[cell_index].get("cell_type") != "code":
        print(f"⚠️ No code cell found at index {cell_index} in {ipynb_path}.")
        return

    # Extract the code from the target cell
    cell_code = cells[cell_index].get("source", [])
    if isinstance(cell_code, list):
        cell_code = "".join(cell_code)

    # Read the target Python file
    if not os.path.exists(py_path):
        print(f"⚠️ Target Python file {py_path} not found.")
        return

    with open(py_path, "r", encoding="utf-8") as f:
        py_lines = f.readlines()

    # Insert the extracted code after the placeholder
    output_lines = []
    inserted = False
    for line in py_lines:
        output_lines.append(line)
        if (not inserted) and (insert_after_tag in line):
            #output_lines.append("\n# --- Injected code from notebook cell ---\n")
            output_lines.append(cell_code + "\n")
            inserted = True

    if not inserted:
        print(f"⚠️ Tag '{insert_after_tag}' not found in {py_path}, skipping injection.")
        return

    # Write back to the Python file
    with open(py_path, "w", encoding="utf-8") as f:
        f.writelines(output_lines)

    print(f"✅ Injected code from cell {cell_index} of {ipynb_path} into {py_path} after '{insert_after_tag}'.")



# =========================================================
# FUNCTION: inject_path_in_behavior_scoring_run
# Replaces a placeholder path in BehaviorScoringRun.ipynb
# with the actual PathConfig.py path from the experiment.
# =========================================================

def inject_path_in_behavior_scoring_run(notebook_path, experimental_folder):
    """
    Replace placeholder path in BehaviorScoringRun.ipynb with:
    {experimental_folder}Protocols/Codes/Config/PathConfig.py
    """
    if not os.path.exists(notebook_path):
        print(f"⚠️ Notebook {notebook_path} not found, skipping injection.")
        return

    with open(notebook_path, "r", encoding="utf-8") as f:
        content = f.read()

    placeholder = "__PLACEHOLDER_PATHCONFIG_ROOT__"
    new_path = os.path.join(experimental_folder, "Protocols/Codes/Config/PathConfig.py")
    new_path = new_path.replace("\\", "/")  # ensure forward slashes for Colab

    if placeholder not in content:
        print(f"⚠️ Placeholder path '{placeholder}' not found in notebook.")
        return

    updated_content = content.replace(placeholder, new_path)

    with open(notebook_path, "w", encoding="utf-8") as f:
        f.write(updated_content)

    print(f"✅ Path injected into {notebook_path}: {new_path}")


# =========================================================
# FUNCTION: ensure_init_py_for_run
# Ensures that a __init__.py exists in the run folders(BehaviorScoring or CreateDataFrames).
# If missing, copies from Config folder (if available).
# =========================================================

def ensure_init_py_for_run(experimental_folder, run_key):
    base_codes_path = os.path.join(experimental_folder, 'Protocols', 'Codes')
    run_folder_name = None

    if run_key == 'BEHAVIOR_SCORING_RUN':
        run_folder_name = "BehaviorScoring"
    elif run_key == 'CREATE_DATAFRAMES_RUN':
        run_folder_name = "CreateDataFrames"
    else:
        print(f"⚠️ Run key {run_key} is not supported for __init__.py check")
        return

    run_folder_path = os.path.join(base_codes_path, run_folder_name)
    config_init_path = os.path.join(base_codes_path, "Config", "__init__.py")
    target_init_path = os.path.join(run_folder_path, "__init__.py")

    # If __init__.py does not exist in the run folder
    if not os.path.exists(target_init_path):
        if os.path.exists(config_init_path):
            shutil.copy2(config_init_path, target_init_path)
            print(f"✅ Copied __init__.py from Config to {run_folder_name}")
        else:
            print(f"⚠️ __init__.py not found in Config to copy to {run_folder_name}, skipping.")
    else:
        print(f"✔️ __init__.py already exists in {run_folder_name}, nothing to do.")


# =========================================================
# MAIN CALLBACK FUNCTION: on_continue_clicked
# This is triggered when the "Continue" button is clicked.
# It:
#   - Validates selected folder
#   - Clones repo
#   - Creates folder structure
#   - Copies required files for each run
#   - Ensures __init__.py exists where needed
#   - Injects paths and code where placeholders exist
# =========================================================

def on_continue_clicked(b):
    with output:
        clear_output()

        global selected_experiment_folder
        if not selected_experiment_folder or not os.path.isdir(selected_experiment_folder):
            print("⚠️ Please select a valid experiment folder before continuing.")
            return

        experimental_folder = selected_experiment_folder
        print(f"✅ Folder selected: {experimental_folder}")

        clone_repo_once(experimental_folder)

        create_experiment_structure(experimental_folder)

        for run_key, run_info in DEPENDENCIES.items():
            if run_info['flag']:
                print(f"\nProcessing {run_key} ...")
                copy_files_for_run(experimental_folder, run_info)
                # Ensure __init__.py exists in the run folder, copy from Config if missing
                ensure_init_py_for_run(experimental_folder, run_key)
            else:
                print(f"\nSkipping {run_key} because its flag is False.")

        # Clone BehaviorScoringRun.ipynb inside Protocols/Codes folder (before injection)
        clone_behavior_scoring_notebook(experimental_folder)

        path_config_path = os.path.join(experimental_folder, "Protocols", "Codes", "Config", "PathConfig.py")
        inject_exp_root_in_pathconfig(path_config_path, experimental_folder)

        py_path = os.path.join(experimental_folder, "Protocols", "Codes", "Config", "ExperimentConfig.py")

        behavior_scoring_notebook_path = os.path.join(experimental_folder, "Protocols", "Codes", "BehaviorScoringRun.ipynb")

        inject_path_in_behavior_scoring_run(behavior_scoring_notebook_path, experimental_folder)

        inject_code_from_ipynb(
              ipynb_path="/content/drive/MyDrive/Matheus_e_Rodrigo/TestRun/_GenerateExperiment.ipynb", ## Just for DEV purposes in the final version this will be changed. THE GENERATE EXPERIMENT MAY HAVE A DEFINED ROOT
              py_path=f"{chooser.selected}Protocols/Codes/Config/ExperimentConfig.py",
              cell_index=5,
              insert_after_tag="#%% CELL 02 – EXPERIMENT CONFIG"
        )

        print("\n✅ All done!")


# Attach the click event handler
continue_button.on_click(on_continue_clicked)